# DROZY Dataset – 5-Fold LSTM Training

Trains the **same CNN-LSTM architecture** used for UTA on the DROZY dataset.

| Setting | Value |
|---|---|
| Embedding dim | 512 |
| LSTM hidden | 64 |
| LSTM layers | 2 |
| Dropout | 0.5 |
| Window size | 30 frames |
| Stride | 10 frames |
| Loss | CrossEntropy (label_smoothing=0.1) |
| Optimizer | Adam (lr=1e-4, wd=1e-5) |
| Patience | 10 epochs |

**Development subjects**: 1, 2, 3, 4, 5, 6, 8  
**Held-out test subjects**: 11, 14

In [ ]:
import sys
import json
import random
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

from sklearn.model_selection import KFold
from sklearn.metrics import (
    accuracy_score, f1_score,
    classification_report, confusion_matrix
)
import seaborn as sns
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

# Reuse identical LSTM model class from UTA – architecture must be same
from src.lstm_model import DrowsinessLSTM

print("PROJECT_ROOT:", PROJECT_ROOT)

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

In [ ]:
DROZY_EMB_DIR = PROJECT_ROOT / "processed" / "DROZY" / "normalized_embeddings"
MODELS_DIR    = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

DEV_SUBJECTS  = [1, 2, 3, 4, 5, 6, 8]  # 5-fold CV
TEST_SUBJECTS = [11, 14]                 # held-out

WINDOW_SIZE = 30
STRIDE      = 10
LABEL_MAP   = {"alert": 0, "low_vigilant": 1, "drowsy": 2}

print("Development subjects:", DEV_SUBJECTS)
print("Test subjects       :", TEST_SUBJECTS)

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)

folds = []
dev_list = list(DEV_SUBJECTS)

for train_idx, val_idx in kf.split(dev_list):
    folds.append({
        "train_subjects": [dev_list[i] for i in train_idx],
        "val_subjects"  : [dev_list[i] for i in val_idx]
    })

for i, fold in enumerate(folds):
    print(f"Fold {i+1}  train={fold['train_subjects']}  val={fold['val_subjects']}")

out_dir = PROJECT_ROOT / "processed" / "DROZY"
out_dir.mkdir(parents=True, exist_ok=True)
with open(out_dir / "folds.json", "w") as f:
    json.dump(folds, f, indent=2)
print("\nFolds saved to processed/DROZY/folds.json")

In [ ]:
def make_windows(subject_ids, emb_dir, window_size=30, stride=10):
    """Creates DataFrame of sliding-window records for given subjects."""
    records = []
    for subj in subject_ids:
        subj_dir = emb_dir / str(subj)
        for condition, label in LABEL_MAP.items():
            npy_path = subj_dir / f"{condition}.npy"
            if not npy_path.exists():
                print(f"  WARNING: {npy_path} not found")
                continue
            emb = np.load(npy_path)  # (T, 512)
            T   = len(emb)
            for start in range(0, T - window_size + 1, stride):
                records.append({
                    "subject"        : subj,
                    "condition"      : condition,
                    "embedding_path" : str(npy_path),
                    "start_idx"      : start,
                    "label"          : label
                })
    return pd.DataFrame(records)

# Sanity check – all subjects
all_df = make_windows(DEV_SUBJECTS + TEST_SUBJECTS, DROZY_EMB_DIR)
print("Total windows:", len(all_df))
print(all_df.groupby(["subject","condition"]).size().unstack())

In [ ]:
class DROZYDataset(Dataset):
    """Sliding-window sequence dataset for DROZY embeddings."""
    def __init__(self, dataframe, window_size=30):
        self.df = dataframe.reset_index(drop=True)
        self.window_size = window_size
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        emb   = np.load(row['embedding_path'])
        start = row['start_idx']
        x = emb[start : start + self.window_size]
        y = row['label']
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.long)

# Quick sanity check
tmp_ds = DROZYDataset(all_df[all_df['subject'] == 1])
x0, y0 = tmp_ds[0]
print("x shape:", x0.shape, " y:", y0.item())

In [ ]:
all_acc, all_f1, all_low_f1, all_reports = [], [], [], []

for fold_idx, fold in enumerate(folds):
    torch.cuda.empty_cache()
    print('='*60)
    print(f"FOLD {fold_idx+1}/5")
    print(f"  Train subjects: {fold['train_subjects']}")
    print(f"  Val   subjects: {fold['val_subjects']}")
    print('='*60)

    train_df = make_windows(fold['train_subjects'], DROZY_EMB_DIR, WINDOW_SIZE, STRIDE)
    val_df   = make_windows(fold['val_subjects'],   DROZY_EMB_DIR, WINDOW_SIZE, STRIDE)
    print(f"  Train windows: {len(train_df)} | Val windows: {len(val_df)}")

    train_loader = DataLoader(DROZYDataset(train_df, WINDOW_SIZE), batch_size=128, shuffle=True,  num_workers=0)
    val_loader   = DataLoader(DROZYDataset(val_df,   WINDOW_SIZE), batch_size=128, shuffle=False, num_workers=0)

    model     = DrowsinessLSTM().to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)

    best_f1, best_acc, best_low_f1, best_report = 0.0, 0.0, 0.0, None
    counter, patience = 0, 10

    for epoch in range(50):
        print(f"\n  Epoch {epoch+1}/50")

        # ── Train ──────────────────────────────────────────────
        model.train()
        tr_p, tr_l = [], []
        for x, y in tqdm(train_loader, leave=False):
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x)
            loss   = criterion(logits, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            preds = logits.argmax(1)
            tr_p.extend(preds.cpu().numpy())
            tr_l.extend(y.cpu().numpy())
        tr_acc = accuracy_score(tr_l, tr_p)
        tr_f1  = f1_score(tr_l, tr_p, average='macro')

        # ── Validate ───────────────────────────────────────────
        model.eval()
        vl_p, vl_l = [], []
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                logits = model(x)
                preds  = logits.argmax(1)
                vl_p.extend(preds.cpu().numpy())
                vl_l.extend(y.cpu().numpy())
        vl_acc = accuracy_score(vl_l, vl_p)
        vl_f1  = f1_score(vl_l, vl_p, average='macro')
        rep    = classification_report(vl_l, vl_p,
                     target_names=['Alert','Low Vigilant','Drowsy'], output_dict=True)
        low_f1 = rep['Low Vigilant']['f1-score']

        print(f"    Train Acc={tr_acc:.4f} F1={tr_f1:.4f} | Val Acc={vl_acc:.4f} F1={vl_f1:.4f} LowVig={low_f1:.4f}")

        # ── Early stopping ──────────────────────────────────────
        if vl_f1 > best_f1:
            best_f1, best_acc, best_low_f1, best_report = vl_f1, vl_acc, low_f1, rep
            counter = 0
            torch.save(model.state_dict(), MODELS_DIR / f'drozy_lstm_fold_{fold_idx+1}.pth')
            print('    Best model saved.')
        else:
            counter += 1
            print(f'    No improvement ({counter}/{patience})')
            if counter >= patience:
                print('    Early stopping.')
                break

    print(f"\n  Fold {fold_idx+1}: Acc={best_acc:.4f}  MacroF1={best_f1:.4f}  LowVigF1={best_low_f1:.4f}")
    all_acc.append(best_acc)
    all_f1.append(best_f1)
    all_low_f1.append(best_low_f1)
    all_reports.append(best_report)

In [ ]:
rows = []
for i, rep in enumerate(all_reports):
    rows.append({
        'Fold'           : i+1,
        'Accuracy'       : all_acc[i],
        'Macro_F1'       : all_f1[i],
        'Alert_F1'       : rep['Alert']['f1-score'],
        'Low_Vigilant_F1': rep['Low Vigilant']['f1-score'],
        'Drowsy_F1'      : rep['Drowsy']['f1-score']
    })

res_df = pd.DataFrame(rows)
mean_r = {'Fold':'Mean', **{c: res_df[c].mean() for c in res_df.columns[1:]}}
std_r  = {'Fold':'Std',  **{c: res_df[c].std()  for c in res_df.columns[1:]}}
res_df = pd.concat([res_df, pd.DataFrame([mean_r, std_r])], ignore_index=True)

print('='*60)
print('DROZY 5-FOLD CROSS-VALIDATION RESULTS')
print('='*60)
print(res_df.to_string(index=False))

res_df.to_csv(PROJECT_ROOT / 'drozy_results_5fold.csv', index=False)
print('\nSaved: drozy_results_5fold.csv')

In [ ]:
best_fold_idx = int(np.argmax(all_f1))
print(f'Best fold: {best_fold_idx+1}  (Macro F1 = {all_f1[best_fold_idx]:.4f})')

fold_best = folds[best_fold_idx]
val_df_b  = make_windows(fold_best['val_subjects'], DROZY_EMB_DIR, WINDOW_SIZE, STRIDE)
val_ldr_b = DataLoader(DROZYDataset(val_df_b, WINDOW_SIZE), batch_size=128, shuffle=False, num_workers=0)

model_best = DrowsinessLSTM().to(device)
model_best.load_state_dict(
    torch.load(MODELS_DIR / f'drozy_lstm_fold_{best_fold_idx+1}.pth', map_location=device)
)
model_best.eval()

vl_p2, vl_l2 = [], []
with torch.no_grad():
    for x, y in val_ldr_b:
        logits = model_best(x.to(device))
        vl_p2.extend(logits.argmax(1).cpu().numpy())
        vl_l2.extend(y.numpy())

CLASS_NAMES = ['Alert', 'Low Vigilant', 'Drowsy']
cm = confusion_matrix(vl_l2, vl_p2)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title(f'DROZY – Best CV Fold ({best_fold_idx+1})')
plt.ylabel('True'); plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig(PROJECT_ROOT / f'drozy_confusion_fold_{best_fold_idx+1}.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
print('='*60)
print('HELD-OUT TEST EVALUATION  (Subjects 11, 14)')
print('='*60)

test_df = make_windows(TEST_SUBJECTS, DROZY_EMB_DIR, WINDOW_SIZE, STRIDE)
print(f'Test windows: {len(test_df)}')
print(test_df.groupby(['subject','condition']).size().unstack())

test_ldr = DataLoader(DROZYDataset(test_df, WINDOW_SIZE), batch_size=128, shuffle=False, num_workers=0)

ts_p, ts_l = [], []
with torch.no_grad():
    for x, y in test_ldr:
        logits = model_best(x.to(device))
        ts_p.extend(logits.argmax(1).cpu().numpy())
        ts_l.extend(y.numpy())

test_acc = accuracy_score(ts_l, ts_p)
test_f1  = f1_score(ts_l, ts_p, average='macro')
print(f'\nTest Accuracy : {test_acc:.4f}')
print(f'Test Macro F1 : {test_f1:.4f}\n')
print(classification_report(ts_l, ts_p, target_names=CLASS_NAMES, digits=4))

In [ ]:
cm_test = confusion_matrix(ts_l, ts_p)
plt.figure(figsize=(6,5))
sns.heatmap(cm_test, annot=True, fmt='d', cmap='Oranges',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('DROZY – Held-Out Test (Subjects 11, 14)')
plt.ylabel('True'); plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'drozy_confusion_heldout.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: drozy_confusion_heldout.png')

In [ ]:
for subj in TEST_SUBJECTS:
    sdf = test_df[test_df['subject'] == subj]
    sldr = DataLoader(DROZYDataset(sdf, WINDOW_SIZE), batch_size=128, shuffle=False, num_workers=0)
    sp, sl = [], []
    with torch.no_grad():
        for x, y in sldr:
            logits = model_best(x.to(device))
            sp.extend(logits.argmax(1).cpu().numpy())
            sl.extend(y.numpy())
    print(f'Subject {subj}: Acc={accuracy_score(sl,sp):.4f}  MacroF1={f1_score(sl,sp,average="macro"):.4f}')
    print(classification_report(sl, sp, target_names=CLASS_NAMES, digits=4))

In [ ]:
summary_rows = []
for i in range(5):
    summary_rows.append({'Split': f'CV Fold {i+1}', 'Accuracy': all_acc[i],
                         'Macro_F1': all_f1[i], 'Low_Vigilant_F1': all_low_f1[i]})
summary_rows.append({'Split':'CV Mean','Accuracy':np.mean(all_acc),
                      'Macro_F1':np.mean(all_f1),'Low_Vigilant_F1':np.mean(all_low_f1)})
summary_rows.append({'Split':'CV Std', 'Accuracy':np.std(all_acc),
                      'Macro_F1':np.std(all_f1), 'Low_Vigilant_F1':np.std(all_low_f1)})
summary_rows.append({'Split':'Held-Out','Accuracy':test_acc,
                      'Macro_F1':test_f1,'Low_Vigilant_F1':f1_score(ts_l,ts_p,average=None)[1]})

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))
summary_df.to_csv(PROJECT_ROOT / 'drozy_final_summary.csv', index=False)
print('\nSaved: drozy_final_summary.csv')